Продолжение рассмотрения инженерного алгоритма, используемого в численном   электродинамическом расчёте волнового двигателя.  Основная тестовая и расчётная  среда — файл MenDrive_real.ipynb.     Исследование будем проводить в несколько этапов.  

Этап 1.
По Bx я наблюдаю неплохую стыковку граничных условий 
(точнее нестыковка по Bx согласно ячейке "execution_count": 482 
достаточно мала).
Проанализируй критически, как может согласоваться существенная 
div_B и очень малая нестыковка по нормальной компоненте магнитной индукции.

Отличный вопрос, Ольга. Прежде чем отвечать, я тщательно проследил всю цепочку определений полей и дисперсионных соотношений в ноутбуке. Вот результат анализа.

---

## Суть парадокса: алгебраическое условие vs дифференциальное

Стыковка нормальной компоненты Bx на границе — это **алгебраическое (поточечное) условие**:

$$B_x^{\rm conductor}(x=\pm a,\; y) = B_x^{\rm vacuum}(x=\pm a,\; y)$$

А div **B** = 0 — это **дифференциальное (объёмное) условие**:

$$\frac{\partial B_x}{\partial x} + \frac{\partial B_y}{\partial y} + \frac{\partial B_z}{\partial z} = 0$$

Это принципиально разные вещи, и вот конкретный механизм их расхождения в текущем алгоритме.

---

## Прямое вычисление: откуда берётся div B ≠ 0

Я проследил компоненты B в проводнике (ячейка index=396):

| Компонента | Формула | y-зависимость | Знаменатель |
|---|---|---|---|
| $B_x$ | $\propto K \cdot \mu_{xx} \cdot \sin(k_y^H y)$ | sin | $c^2k_z^2 - \varepsilon_{yy}\mu_{xx}\omega^2$ |
| $B_y$ | $\propto k_y^H \cdot \mu_{yy} \cdot \cos(k_y^H y)$ | cos | $c^2k_z^2 - \varepsilon_{xx}\mu_{yy}\omega^2$ |
| $B_z$ | $\propto \mu_{zz} \cdot \sin(k_y^H y)$ | sin | — |

Все три имеют одинаковую x-зависимость $\exp(-iK_{\rm cond} \cdot \text{sign} \cdot (x+a))$.

Вычисляя div B для **изотропного** проводника ($\varepsilon_{xx}=\varepsilon_{yy}=\varepsilon$, $\mu_{xx}=\mu_{yy}=\mu_{zz}=\mu$), получаем:

$$\text{div}\,\mathbf{B} \;\propto\; i\,k_z\,\sin(k_y^H y) \;\cdot\; \frac{c^2(K^2+k_y^{H\,2}) + c^2k_z^2 - \varepsilon\mu\omega^2}{c^2k_z^2 - \varepsilon\mu\omega^2}$$

Числитель обращается в ноль тогда и только тогда, когда:

$$K^2 + k_y^{H\,2} + k_z^2 = \frac{\varepsilon\mu\omega^2}{c^2}$$

## Где корень проблемы

Дисперсионное соотношение в вакууме (ячейка index=437) **содержит** $k_y^{H\,2}$:

$$K_{\rm vac}^H = \sqrt{\kappa_{\rm vac}^2 - k_z^2 - k_y^{H\,2}} = \sqrt{\frac{\omega^2}{c^2} - k_z^2 - k_y^{H\,2}}$$

→ $K_{\rm vac}^2 + k_y^{H\,2} + k_z^2 = \omega^2/c^2$ → **div B = 0 в вакууме** ✓

А дисперсионное соотношение в проводнике (ячейка index=455, изотропная ветвь) **не содержит** $k_y^{H\,2}$:

$$K_{\rm cond}^2 = \left(\frac{\omega^2}{c^2}\,\varepsilon_{xx}\mu_{xx} - k_z^2\right)\cdot\mu_k$$

Для изотропного случая ($\mu_k = \mu_{zz}/\mu_{xx} = 1$):

$$K_{\rm cond}^2 = \frac{\varepsilon\mu\omega^2}{c^2} - k_z^2$$

Подставляя в числитель:

$$c^2 K_{\rm cond}^2 + c^2 k_y^{H\,2} + c^2 k_z^2 - \varepsilon\mu\omega^2 = \mathbf{c^2 k_y^{H\,2}}$$

Итого:

$$\boxed{\text{div}\,\mathbf{B}_{\rm conductor} = i\,k_z\,\sin(k_y^H y) \;\cdot\; \frac{c^2\,k_y^{H\,2}}{c^2k_z^2 - \varepsilon\mu\omega^2} \cdot B_{H,zl}\,e^{g(x)}}$$

Дивергенция пропорциональна $k_y^{H\,2}$. При $m=0$, $b=5$ см:  $k_y^H = \pi/(2b) = \pi/10 \approx 0.314$ см⁻¹, и эта величина отлична от нуля.

---

## Почему Bx при этом стыкуется хорошо

На границе $x = -a$:

- $B_x^{\rm vac}(-a,\,y) = f_{\rm vac}(K_{\rm vac},\,A_1,\,A_2) \cdot \sin(k_y^H y)$
- $B_x^{\rm cond}(-a,\,y) = f_{\rm cond}(K_{\rm cond},\,B_{H,zl}) \cdot \sin(k_y^H y)$

Зависимость от y **одинаковая** ($\sin(k_y^H y)$), поэтому она сокращается, и условие сводится к равенству **скалярных амплитуд** — обычному алгебраическому уравнению. Более того, Bx выражается через ту же функцию Hz (а именно, через $\partial H_z/\partial x$), которая и без того хорошо стыкуется, ибо SVD оптимизирует именно Hz и Ey.

Иначе говоря: Bx «наследует» качество стыковки от Hz — это **та же информация**, вид сбоку.

А div B — это **новая** информация: она проверяет согласованность **производных по всем трём координатам**, и именно здесь «всплывает» пропущенное $k_y^{H\,2}$ в дисперсионном соотношении проводника.

---

## Резюме

Хорошая стыковка Bx и большой div B — это **не** противоречие, а закономерное следствие одной и той же структурной особенности: дисперсионное соотношение в проводнике не учитывает $k_y^{H\,2}$, что не мешает поточечной стыковке (Bx), но нарушает дифференциальную самосогласованность (div B = 0).

